# Voice Metrics Evaluation

Evaluate the voice emotion model and export metrics to `reports/metrics_voice.csv`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from app.models.voice.emotion_train import gather_features, FEATURE_COUNT, MODEL_PATH


In [ ]:
data = gather_features(limit_per_class=0, min_per_class=25)
df = pd.DataFrame(data)
X = df[[f'f{i}' for i in range(FEATURE_COUNT)]].values
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
else:
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=25, random_state=42, n_jobs=-1))
    ])
    model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
rec = recall_score(y_test, y_pred, average='macro', zero_division=0)

roc = None
if hasattr(model, 'predict_proba'):
    proba = model.predict_proba(X_test)
    classes = np.unique(y)
    try:
        y_bin = label_binarize(y_test, classes=classes)
        roc = roc_auc_score(y_bin, proba, average='macro', multi_class='ovr')
    except Exception:
        roc = None

rows = [
    {'Metric': 'accuracy', 'mean': float(acc), 'std': ''},
    {'Metric': 'f1', 'mean': float(f1), 'std': ''},
    {'Metric': 'precision', 'mean': float(prec), 'std': ''},
    {'Metric': 'recall', 'mean': float(rec), 'std': ''},
]
if roc is not None:
    rows.append({'Metric': 'roc_auc', 'mean': float(roc), 'std': ''})

report_path = Path('reports') / 'metrics_voice.csv'
df_new = pd.DataFrame(rows)
if report_path.exists():
    df_old = pd.read_csv(report_path)
    if 'std' not in df_old.columns:
        df_old['std'] = ''
    df_old = df_old[~df_old['Metric'].isin(df_new['Metric'])]
    df_out = pd.concat([df_old, df_new], ignore_index=True)
else:
    df_out = df_new
report_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(report_path, index=False)
print('Saved metrics to', report_path)
